Steps to be Performed
======================

1. install all the  required libraries

2. import the required libraries

3. load the pre trained model

4. Prepare documents

5. convert document into embeddings

6. store embeddings into FAISS index

7. retrieve the top k relevant documents

8. generate final answers


In [1]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 120.8 MB/s eta 0:00:00


In [2]:
pip install transformers

In [3]:
#import the library
import torch
from transformers import DPRQuestionEncoder,DPRQuestionEncoderTokenizer,DPRContextEncoder,DPRContextEncoderTokenizer
from transformers import pipeline,AutoModelForSeq2SeqLM,AutoTokenizer
import faiss
import numpy as np

In [4]:
#load the huggingface model
question_encoder=DPRQuestionEncoder.from_pretrained('facebook/dpr-question_encoder-single-nq-base')
question_tokenizer=DPRQuestionEncoderTokenizer.from_pretrained('facebook/dpr-question_encoder-single-nq-base')
context_encoder=DPRContextEncoder.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
context_tokenizer=DPRContextEncoderTokenizer.from_pretrained('facebook/dpr-ctx_encoder-single-nq-base')
rag_model=AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
rag_tokenizer=AutoTokenizer.from_pretrained('google/flan-t5-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/493 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [5]:
documents = [
    "Non-Disclosure Agreement (NDA): A contract that ensures confidential information shared between parties is not disclosed to unauthorized third parties.",
    "Service Level Agreement (SLA): A contract between a service provider and client defining the expected level of service, performance metrics, and remedies for non-compliance.",
    "Employment Agreement: A legally binding contract that outlines the terms and conditions of employment, including job role, compensation, and termination clauses.",
    "Partnership Agreement: A document that governs the relationship between two or more partners in a business, defining profit-sharing, responsibilities, and dispute resolution.",
    "Lease Agreement: A contract between a landlord and tenant granting the tenant the right to use property for a specified time in exchange for rent.",
    "Franchise Agreement: A contract that allows one party (franchisee) to use the brand, business model, and intellectual property of another party (franchisor) in exchange for fees or royalties.",
    "Sales Agreement: A legally binding contract between a buyer and seller that defines the terms of the sale of goods or services.",
    "Loan Agreement: A contract between a lender and borrower specifying the loan amount, repayment schedule, interest rate, and default consequences.",
    "Joint Venture Agreement: A contract between two or more parties to collaborate on a specific project while sharing profits, losses, and responsibilities.",
    "Settlement Agreement: A legal document that resolves a dispute between parties without admitting fault, usually involving compensation or corrective action."
]


In [6]:
#documents into embedding
doc_embeddings=[]
for doc in documents:
  inputs=context_tokenizer(doc,return_tensors='pt',padding=True)
  embedding=context_encoder(**inputs).pooler_output.detach().numpy()
  doc_embeddings.append(embedding)
doc_embeddings=np.vstack(doc_embeddings)


In [7]:
doc_embeddings

array([[-0.05021099, -0.28506443,  0.06728801, ..., -0.32398424,
        -0.47654924,  0.36502886],
       [ 0.34969273, -0.13094346,  0.38761407, ..., -0.09933607,
        -0.73478746, -0.44987452],
       [ 0.46302837,  0.23219466,  0.3512156 , ..., -0.45680076,
        -0.37783754,  0.28486365],
       ...,
       [ 0.14004833,  0.00571589,  0.7082764 , ..., -0.21053351,
        -0.564824  ,  0.2681207 ],
       [ 0.446375  ,  0.03985277, -0.07466862, ..., -0.59789485,
        -0.18449621,  0.07993262],
       [-0.40429395,  0.15212828,  0.28268966, ..., -0.10031056,
        -0.40978715,  0.21511689]], dtype=float32)

In [8]:
doc_embeddings.shape[1]

768

In [9]:
#create faiss index for fast retrieval
dimension=doc_embeddings.shape[1]
faiss_index=faiss.IndexFlatL2(dimension)
faiss_index.add(doc_embeddings)

In [10]:
#query processing and retrieval process
def retrieve_top_k(query,k=2):
  query_inputs=question_tokenizer(query,return_tensors='pt')
  query_embeddings=question_encoder(**query_inputs).pooler_output.detach().numpy()
  distances,indices=faiss_index.search(query_embeddings,k)
  retrieved_docs=[documents[i] for i in indices[0]]
  return retrieved_docs

In [11]:
#generate the response using RAG
def generate_response(query):
  retrieved_docs=retrieve_top_k(query,k=2)
  context=" ".join(retrieved_docs)
  inputs=rag_tokenizer(f"Question:{query} context:{context}",return_tensors='pt')
  output=rag_model.generate(**inputs)
  response=rag_tokenizer.decode(output[0],skip_special_tokens=True)
  return response

In [12]:
#question answer bot
def chat():
  print('Hi Ask me anything or type stop to end the conversation')
  while True:
    query=input('you: ')
    if query.lower()=='stop':
      print('Goodbye')
      break
    response=generate_response(query)
    print(f'GPT: {response}')

In [14]:
chat()

Hi Ask me anything or type stop to end the conversation
you: What is SLA here?
GPT: Service Level Agreement
you: What is that?
GPT: Joint Venture Agreement
you: stop
Goodbye
